# 02 — Fine-tuned baselines
Models (each × 4 tasks × 5 seeds, reported as mean ± std macro-F1):
- **Image-MLP (CLIP)** — MLP on frozen CLIP ViT-L/14 features from notebook 01
- **Text-FT BanglaBERT / XLM-R** — full fine-tuning of the text encoder
- **Fusion BanglaBERT+CLIP / XLM-R+CLIP** — fine-tuned text encoder + projected CLIP image feature → MLP head

Kernel: **Python (internship1)**. Requires `features/*_img.npy` from notebook 01.

**Runtime:** ~1.5 h per seed for all text/fusion runs on an RTX 3050 (≈7 h for 5 seeds).
Runs are logged one by one and **skipped if already done**, so you can stop and resume anytime.
Tip: run with `SEEDS = [42]` first, check results, then add the other seeds.

## 1. Imports & config

In [ ]:
import os, gc, json, time, random, platform, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import transformers, sklearn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score

ROOT = "/mnt/f415dcc8-27a5-4e66-8493-25f1d939974c/internshipwork/journal"
P, FEAT = f"{ROOT}/processed", f"{ROOT}/features"
RES = f"{ROOT}/results/02_finetuned"
PRED = f"{RES}/preds"
os.makedirs(PRED, exist_ok=True)

DATASETS = ["bhm", "banglaabusememe", "mimosa"]
TASKS = [("bhm", "label"), ("banglaabusememe", "label"), ("mimosa", "label"), ("mimosa", "label_bin")]
SEEDS = [42, 43, 44, 45, 46]          # start with [42] to check timing
TEXT_MODELS = {"BanglaBERT": "csebuetnlp/banglabert", "XLM-R": "xlm-roberta-base"}

# text / fusion training
MAX_LEN, BS, EVAL_BS = 128, 16, 64
MAX_EPOCHS, PATIENCE = 5, 2
LR_ENC, LR_HEAD, WD, WARMUP, DROPOUT = 2e-5, 5e-4, 0.01, 0.1, 0.1
# image-MLP training
IMG_LR, IMG_BS, IMG_EPOCHS, IMG_PATIENCE = 1e-3, 64, 50, 5

dev = "cuda"
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| AMP:", AMP_DTYPE)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 2. Load data + frozen CLIP image features

In [ ]:
data = {}
for ds in DATASETS:
    df = pd.read_csv(f"{P}/{ds}.csv").fillna("")
    img = np.load(f"{FEAT}/{ds}_img.npy").astype(np.float32)
    assert len(df) == len(img), f"row mismatch in {ds}"
    data[ds] = {"df": df, "img": img}
    print(ds, df.shape, img.shape)

def idx_of(df, s):
    return np.where(df["split"].values == s)[0]

# save ground-truth test labels once (for later analysis)
for ds, lab in TASKS:
    df = data[ds]["df"]
    np.save(f"{PRED}/{ds}__{lab}__ytrue.npy", df[lab].values[idx_of(df, "test")].astype(int))

## 3. Run logging (resume-safe)

In [ ]:
RUNS_CSV = f"{RES}/runs.csv"
done = set(pd.read_csv(RUNS_CSV)["run_id"]) if os.path.exists(RUNS_CSV) else set()
print("runs already done:", len(done))

def slug(s):
    return s.replace(" ", "_").replace("+", "_").replace("(", "").replace(")", "")

def class_weights(y, n):
    counts = np.bincount(y, minlength=n)
    return torch.tensor(len(y) / (n * np.maximum(counts, 1)), dtype=torch.float32, device=dev)

def record(run_id, ds, lab, model_name, seed, val_f1, best_ep, prob, y_te, t0):
    np.save(f"{PRED}/{run_id}.npy", prob)
    pred = prob.argmax(1)
    row = {"run_id": run_id, "dataset": ds, "task": lab, "model": model_name, "seed": seed,
           "best_epoch": best_ep, "val_f1": val_f1,
           "test_f1": f1_score(y_te, pred, average="macro"),
           "test_acc": accuracy_score(y_te, pred),
           "minutes": round((time.time() - t0) / 60, 2)}
    pd.DataFrame([row]).to_csv(RUNS_CSV, mode="a", header=not os.path.exists(RUNS_CSV), index=False)
    done.add(run_id)
    print(f"{ds:16s} {lab:10s} {model_name:24s} seed {seed}  val {val_f1:.4f}  test {row['test_f1']:.4f}  ({row['minutes']} min)")

## 4. Image-MLP on frozen CLIP features

In [ ]:
class ImgMLP(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, n))
    def forward(self, x):
        return self.net(x)

def run_image_mlp(ds, lab, seed):
    set_seed(seed)
    df = data[ds]["df"]; y = df[lab].values.astype(np.int64); n = int(y.max()) + 1
    tr, va, te = [idx_of(df, s) for s in ("train", "val", "test")]
    X = torch.from_numpy(data[ds]["img"]).to(dev); Y = torch.from_numpy(y).to(dev)
    T = lambda a: torch.from_numpy(a).to(dev)

    model = ImgMLP(X.shape[1], n).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=IMG_LR, weight_decay=WD)
    lossf = nn.CrossEntropyLoss(weight=class_weights(y[tr], n))
    g = torch.Generator().manual_seed(seed)

    best_f, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(IMG_EPOCHS):
        model.train()
        perm = torch.from_numpy(tr)[torch.randperm(len(tr), generator=g)].to(dev)
        for i in range(0, len(perm), IMG_BS):
            b = perm[i:i + IMG_BS]
            opt.zero_grad()
            lossf(model(X[b]), Y[b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            f = f1_score(y[va], model(X[T(va)]).argmax(1).cpu().numpy(), average="macro")
        if f > best_f:
            best_f, best_ep, bad = f, ep + 1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= IMG_PATIENCE:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(X[T(te)]), 1).cpu().numpy()
    return best_f, best_ep, prob, y[te]

name = "Image-MLP (CLIP)"
for seed in SEEDS:
    for ds, lab in TASKS:
        run_id = f"{ds}__{lab}__{slug(name)}__s{seed}"
        if run_id in done:
            continue
        t0 = time.time()
        vf, ep, prob, y_te = run_image_mlp(ds, lab, seed)
        record(run_id, ds, lab, name, seed, vf, ep, prob, y_te, t0)

## 5. Text / fusion model

In [ ]:
class MemeDS(Dataset):
    def __init__(self, enc, img, y, idx):
        idx = torch.from_numpy(idx)
        self.ids, self.mask = enc["input_ids"][idx], enc["attention_mask"][idx]
        self.img = torch.from_numpy(img)[idx]
        self.y = torch.from_numpy(y)[idx]
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.ids[i], self.mask[i], self.img[i], self.y[i]

class TextFusion(nn.Module):
    def __init__(self, name, n, img_dim=0):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        h = self.enc.config.hidden_size
        self.use_img = img_dim > 0
        if self.use_img:
            self.img_proj = nn.Sequential(nn.Linear(img_dim, h), nn.GELU(), nn.Dropout(DROPOUT))
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(h * (2 if self.use_img else 1), h),
                                  nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h, n))
    def forward(self, ids, mask, img=None):
        hs = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).to(hs.dtype)
        t = (hs * m).sum(1) / m.sum(1).clamp(min=1)          # mean pooling
        z = torch.cat([t, self.img_proj(img)], -1) if self.use_img else t
        return self.head(z)

_tok_cache = {}
def get_enc(model_key, ds):
    k = (model_key, ds)
    if k not in _tok_cache:
        tok = AutoTokenizer.from_pretrained(TEXT_MODELS[model_key])
        _tok_cache[k] = tok(data[ds]["df"]["text"].astype(str).tolist(), padding="max_length",
                            truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return _tok_cache[k]

@torch.no_grad()
def predict(model, loader, use_img):
    model.eval(); out = []
    for ids, mask, img, _ in loader:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            lo = model(ids.to(dev), mask.to(dev), img.to(dev) if use_img else None)
        out.append(torch.softmax(lo.float(), 1).cpu())
    return torch.cat(out).numpy()

def run_text(model_key, ds, lab, seed, use_img):
    set_seed(seed)
    df = data[ds]["df"]; y = df[lab].values.astype(np.int64); n = int(y.max()) + 1
    enc, img = get_enc(model_key, ds), data[ds]["img"]
    tr, va, te = [idx_of(df, s) for s in ("train", "val", "test")]
    g = torch.Generator().manual_seed(seed)
    dl_tr = DataLoader(MemeDS(enc, img, y, tr), batch_size=BS, shuffle=True, generator=g)
    dl_va = DataLoader(MemeDS(enc, img, y, va), batch_size=EVAL_BS)
    dl_te = DataLoader(MemeDS(enc, img, y, te), batch_size=EVAL_BS)

    model = TextFusion(TEXT_MODELS[model_key], n, img.shape[1] if use_img else 0).to(dev)
    enc_ids = {id(p) for p in model.enc.parameters()}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW([{"params": list(model.enc.parameters()), "lr": LR_ENC},
                             {"params": head_params, "lr": LR_HEAD}], weight_decay=WD)
    steps = len(dl_tr) * MAX_EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    lossf = nn.CrossEntropyLoss(weight=class_weights(y[tr], n))

    best_f, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        for ids, mask, im, yb in tqdm(dl_tr, desc=f"{model_key} {ds} ep{ep + 1}", leave=False):
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                lo = model(ids.to(dev), mask.to(dev), im.to(dev) if use_img else None)
            loss = lossf(lo.float(), yb.to(dev))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        f = f1_score(y[va], predict(model, dl_va, use_img).argmax(1), average="macro")
        if f > best_f:
            best_f, best_ep, bad = f, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    prob = predict(model, dl_te, use_img)
    del model, opt, best_state; gc.collect(); torch.cuda.empty_cache()
    return best_f, best_ep, prob, y[te]

## 6. Run text-only and fusion fine-tuning (long — resumable)

In [ ]:
experiments = [(f"Text-FT {k}", k, False) for k in TEXT_MODELS] + \
              [(f"Fusion {k}+CLIP", k, True) for k in TEXT_MODELS]

for seed in SEEDS:
    for name, key, use_img in experiments:
        for ds, lab in TASKS:
            run_id = f"{ds}__{lab}__{slug(name)}__s{seed}"
            if run_id in done:
                continue
            t0 = time.time()
            vf, ep, prob, y_te = run_text(key, ds, lab, seed, use_img)
            record(run_id, ds, lab, name, seed, vf, ep, prob, y_te, t0)

## 7. Summary table (mean ± std over seeds)

In [ ]:
runs = pd.read_csv(RUNS_CSV)
s = runs.groupby(["dataset", "task", "model"])["test_f1"].agg(["mean", "std", "count"]).reset_index()
s["test_f1"] = s.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}" if r["count"] > 1 else f"{r['mean']:.4f}", axis=1)
table = s.pivot_table(index=["dataset", "task"], columns="model", values="test_f1", aggfunc="first")
print("seeds per cell:", s["count"].min(), "-", s["count"].max())
table

## 8. Save summary + config

In [ ]:
s.to_csv(f"{RES}/summary.csv", index=False)
table.to_csv(f"{RES}/summary_table.csv")

config = {
    "seeds": SEEDS, "tasks": TASKS, "text_models": TEXT_MODELS,
    "image_features": "frozen CLIP ViT-L/14 (notebook 01)",
    "text": {"max_len": MAX_LEN, "batch_size": BS, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
             "lr_encoder": LR_ENC, "lr_head": LR_HEAD, "weight_decay": WD, "warmup": WARMUP,
             "dropout": DROPOUT, "pooling": "mean", "amp": str(AMP_DTYPE)},
    "image_mlp": {"lr": IMG_LR, "batch_size": IMG_BS, "max_epochs": IMG_EPOCHS, "patience": IMG_PATIENCE},
    "loss": "weighted cross-entropy (inverse class frequency on train)",
    "model_selection": "best epoch by validation macro-F1",
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__,
                 "gpu": torch.cuda.get_device_name(0)},
}
with open(f"{RES}/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("saved to", RES)